# 03 — Results

Training curves, the ablation matrix, and sample generations. Reads the committed run log
and ablation results; sample generation runs only if a trained checkpoint is present.

In [ ]:
import sys, json
from pathlib import Path
import matplotlib.pyplot as plt
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
rows = [json.loads(l) for l in open(REPO/'results'/'real_run_metrics_300steps.jsonl') if 'loss/ce' in l]
steps = [r['step'] for r in rows]

In [ ]:
# Training loss + perplexity + LR schedule
fig, ax = plt.subplots(1, 3, figsize=(16,4))
ax[0].plot(steps, [r['loss/ce'] for r in rows], marker='o', ms=3, label='CE')
ax[0].plot(steps, [r['loss/aux'] for r in rows], marker='o', ms=3, label='aux (routing)')
ax[0].set_title('Training loss'); ax[0].set_xlabel('step'); ax[0].legend()
ax[1].plot(steps, [r['train/perplexity'] for r in rows], marker='o', ms=3, c='tab:red'); ax[1].set_yscale('log')
ax[1].set_title('Train perplexity (log)'); ax[1].set_xlabel('step')
ax[2].plot(steps, [r['lr'] for r in rows], marker='o', ms=3, c='tab:green')
ax[2].set_title('LR (warmup + cosine)'); ax[2].set_xlabel('step')
plt.tight_layout(); plt.show()
print('CE: %.2f -> best %.2f | per-batch sawtooth reflects the multi-task mixture' % (rows[0]['loss/ce'], min(r['loss/ce'] for r in rows)))

In [ ]:
# Ablation matrix (committed results)
abl = json.load(open(REPO/'report'/'figures'/'ablation_results.json'))
import pandas as pd
df = pd.DataFrame(abl)
display(df[['variant','perplexity','load_balance','specialization','params_M']])

In [ ]:
# Ablation comparison figure
from IPython.display import Image, display
display(Image(str(REPO/'report'/'figures'/'ablation_comparison.png')))

In [ ]:
# Sample generations per modality (requires a trained checkpoint)
ckpt = REPO/'checkpoints'/'small-moe-baseline'/'final'
if (ckpt/'hf').exists():
    from src.data.tokenizer import build_tokenizer
    from src.eval.generate import generate_one
    from src.model.model import SmallMoE
    tok = build_tokenizer(); model = SmallMoE.from_pretrained(ckpt)
    prompts = {
        'math': 'Question: What is 12 + 15?\nAnswer:',
        'logic': 'All cats are animals. Tom is a cat.\nQuestion: Is Tom an animal?\nA. Yes\nB. No\nAnswer:',
        'code': 'def add(a, b):',
        'language': 'The weather today is',
    }
    for m, p in prompts.items():
        out = generate_one(model, tok, p, m, max_new_tokens=32)
        print(f'[{m}] {p!r}\n   -> {out!r}\n')
else:
    print('No local checkpoint (329MB not in git). Train via notebooks/colab_train.ipynb, then rerun.')